# Advertising Click-Through Rate Prediction

This project predicts whether an online advertisement will be clicked, using the Avazu CTR dataset.

The workflow includes:
- Data loading and cleaning
- Exploratory data analysis
- Feature engineering: one-hot encoding for low-cardinality features, frequency encoding for high-cardinality features
- Handling class imbalance (the dataset has a much lower click rate than non-click rate)
- Logistic Regression baseline and XGBoost model, both imbalance-aware
- Lightweight hyperparameter search for XGBoost
- Model comparison using ROC-AUC, PR-AUC, and Log Loss
- XGBoost feature importance analysis

**Dataset size:** 404,290 advertising impressions

**Data source:** [Avazu Click-Through Rate Prediction (Kaggle)](https://www.kaggle.com/c/avazu-ctr-prediction). Download `train.csv` (or the filtered subset used here) and update `DATA_PATH` below.


## 1. Setup and Data Loading

The dataset is loaded with `id` stored as a string so the identifier is not interpreted as a numeric feature.
The original index-like `Unnamed: 0` column is removed before analysis.

`DATA_PATH` is set relative to the repo (`data/filtered_train.csv`) instead of a personal Google Drive path, so this notebook can be re-run by anyone who clones the repo and drops the CSV into `data/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score, precision_recall_curve

from xgboost import XGBClassifier

RANDOM_STATE = 42

# Update this path if the dataset is stored elsewhere.
# Expected location after cloning this repo: data/filtered_train.csv
DATA_PATH = "data/filtered_train.csv"

df = pd.read_csv(DATA_PATH, dtype={"id": str})

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Dataset shape:", df.shape)
df.head()


## 2. Dataset Overview

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Overall CTR:", df["click"].mean())
print("Unique IDs:", df["id"].nunique())
print("Duplicate IDs:", df["id"].duplicated().sum())

# The class imbalance matters a lot for how we train and evaluate models below.
click_counts = df["click"].value_counts()
print("\nClass balance:")
print(click_counts)
print("Positive (click) rate: {:.2%}".format(df["click"].mean()))


## 3. Exploratory Data Analysis

### CTR by Banner Position

In [ ]:
banner_summary = df.groupby("banner_pos")["click"].agg(["count", "mean"])
banner_summary


In [ ]:
banner_summary["mean"].plot(kind="bar")

plt.title("Click-Through Rate by Banner Position")
plt.xlabel("Banner Position")
plt.ylabel("CTR")
plt.show()


### CTR by Hour of Day

The hour-of-day feature is extracted from the original `hour` field.

In [ ]:
df["hour_of_day"] = df["hour"].astype(str).str[-2:].astype(int)

hour_summary = df.groupby("hour_of_day")["click"].agg(["count", "mean"])
hour_summary


In [ ]:
hour_summary["mean"].plot(kind="line", marker="o")

plt.title("Click-Through Rate by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("CTR")
plt.xticks(range(24))
plt.show()


### Cardinality of Categorical Features

Before deciding how to encode each column, it's worth checking how many unique values each one has.
Low-cardinality columns are safe to one-hot encode. High-cardinality columns (site/app/device identifiers)
would blow up the feature space with one-hot encoding, so they need a different treatment — see Section 4.

In [ ]:
candidate_cols = [
    "site_id", "site_domain", "site_category",
    "app_id", "app_domain", "app_category",
    "device_id", "device_ip", "device_model", "device_type", "device_conn_type",
    "banner_pos", "C1", "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21"
]

cardinality = df[candidate_cols].nunique().sort_values(ascending=False)
cardinality


## 4. Feature Preparation

**Low-cardinality categorical features** (a few to a few hundred unique values) are one-hot encoded, same as before.

**High-cardinality identifier features** (`site_id`, `app_id`, `device_id`, `device_ip`, `device_model`, `site_domain`)
can have thousands to hundreds of thousands of unique values. One-hot encoding these would be both
computationally expensive and prone to overfitting on rare categories. Instead they're **frequency-encoded**:
each category is replaced by how often it appears in the *training* data. This is fit only on the training
set and then applied to the test set, so information about the test set never leaks into training — categories
seen only at test time simply get a frequency of 0.

These identifier features are usually where a lot of the real predictive signal lives in CTR problems (they
capture per-site / per-app / per-device click-propensity), so adding them is expected to meaningfully improve
on the low-cardinality-only baseline from the original version of this notebook.


In [ ]:
df = df.drop(columns=["id", "hour"])

low_card_cols = [
    "site_category",
    "app_category",
    "device_type",
    "device_conn_type",
    "banner_pos",
    "hour_of_day",
    "C1",
    "C15",
    "C16",
    "C18",
]

high_card_cols = [
    "site_id",
    "site_domain",
    "app_id",
    "app_domain",
    "device_id",
    "device_ip",
    "device_model",
    "C14",
    "C17",
    "C19",
    "C20",
    "C21",
]

feature_cols = low_card_cols + high_card_cols

X = df[feature_cols]
y = df["click"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Train CTR:", y_train.mean())
print("Test CTR:", y_test.mean())


In [ ]:
class FrequencyEncoder:
    """Replaces each category with its frequency in the training data.

    Fit only on training data to avoid leakage. Categories unseen during
    fit (e.g. new site_id values at test time) are encoded as 0.
    """

    def __init__(self, cols):
        self.cols = cols
        self.freq_maps_ = {}

    def fit(self, X, y=None):
        for col in self.cols:
            self.freq_maps_[col] = X[col].value_counts(normalize=True)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps_[col]).fillna(0.0)
        return X[self.cols]


freq_encoder = FrequencyEncoder(cols=high_card_cols)
X_train_freq = freq_encoder.fit(X_train).transform(X_train)
X_test_freq = freq_encoder.transform(X_test)

print("Frequency-encoded train shape:", X_train_freq.shape)
X_train_freq.head()


In [ ]:
onehot = OneHotEncoder(handle_unknown="ignore")

X_train_onehot = onehot.fit_transform(X_train[low_card_cols])
X_test_onehot = onehot.transform(X_test[low_card_cols])

from scipy.sparse import hstack, csr_matrix

X_train_processed = hstack([X_train_onehot, csr_matrix(X_train_freq.values)]).tocsr()
X_test_processed = hstack([X_test_onehot, csr_matrix(X_test_freq.values)]).tocsr()

feature_names = list(onehot.get_feature_names_out(low_card_cols)) + high_card_cols

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


## 5. Handling Class Imbalance

The click rate is well under 50%, so a model that ignores this will lean toward predicting "no click" and
still look accurate while being nearly useless at ranking likely clicks. Two adjustments:

- **Logistic Regression**: `class_weight="balanced"` reweights the loss inversely proportional to class frequency.
- **XGBoost**: `scale_pos_weight` (ratio of negative to positive examples) does the equivalent for gradient boosting.

We also report **PR-AUC** (average precision) alongside ROC-AUC. With imbalanced classes, ROC-AUC can look
deceptively good while precision on the minority (click) class is still poor — PR-AUC is more sensitive to that.


In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"Negative examples: {neg}, Positive examples: {pos}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


## 6. Logistic Regression Baseline

In [ ]:
log_model = LogisticRegression(
    solver="saga",
    max_iter=500,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

log_model.fit(X_train_processed, y_train)

log_pred_prob = log_model.predict_proba(X_test_processed)[:, 1]

log_auc = roc_auc_score(y_test, log_pred_prob)
log_pr_auc = average_precision_score(y_test, log_pred_prob)
log_loss_value = log_loss(y_test, log_pred_prob)

print("Logistic Regression ROC-AUC:", log_auc)
print("Logistic Regression PR-AUC:", log_pr_auc)
print("Logistic Regression Log Loss:", log_loss_value)


## 7. XGBoost Model

XGBoost is used as a nonlinear model to capture relationships and interactions that Logistic Regression may
not represent. `scale_pos_weight` is set from the training class balance computed above.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

xgb_pred_prob = xgb_model.predict_proba(X_test_processed)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_pred_prob)
xgb_pr_auc = average_precision_score(y_test, xgb_pred_prob)
xgb_loss_value = log_loss(y_test, xgb_pred_prob)

print("XGBoost ROC-AUC:", xgb_auc)
print("XGBoost PR-AUC:", xgb_pr_auc)
print("XGBoost Log Loss:", xgb_loss_value)


## 8. Lightweight Hyperparameter Search

Rather than hand-picking XGBoost's hyperparameters, a small randomized search is run over the parameters
most likely to matter (tree depth, learning rate, number of trees, subsampling). This uses 3-fold
stratified cross-validation on the training set only, scored on ROC-AUC, so the held-out test set stays
untouched until final evaluation.

This is intentionally a small search (a handful of candidates) to keep runtime reasonable — the goal is to
show a defensible, reproducible tuning process, not to exhaustively grid search.


In [ ]:
param_distributions = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "n_estimators": [200, 300, 400],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
}

base_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train_processed, y_train)

print("Best CV ROC-AUC:", search.best_score_)
print("Best params:", search.best_params_)


In [ ]:
xgb_tuned = search.best_estimator_
xgb_tuned_pred_prob = xgb_tuned.predict_proba(X_test_processed)[:, 1]

xgb_tuned_auc = roc_auc_score(y_test, xgb_tuned_pred_prob)
xgb_tuned_pr_auc = average_precision_score(y_test, xgb_tuned_pred_prob)
xgb_tuned_loss_value = log_loss(y_test, xgb_tuned_pred_prob)

print("Tuned XGBoost ROC-AUC:", xgb_tuned_auc)
print("Tuned XGBoost PR-AUC:", xgb_tuned_pr_auc)
print("Tuned XGBoost Log Loss:", xgb_tuned_loss_value)


## 9. Model Comparison

In [ ]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost (default params)", "XGBoost (tuned)"],
    "ROC-AUC": [log_auc, xgb_auc, xgb_tuned_auc],
    "PR-AUC": [log_pr_auc, xgb_pr_auc, xgb_tuned_pr_auc],
    "Log Loss": [log_loss_value, xgb_loss_value, xgb_tuned_loss_value]
})

model_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_results.plot(x="Model", y="ROC-AUC", kind="bar", legend=False, ax=axes[0], color="steelblue")
axes[0].set_title("ROC-AUC by Model")
axes[0].set_ylabel("ROC-AUC")
axes[0].tick_params(axis="x", rotation=20)

model_results.plot(x="Model", y="PR-AUC", kind="bar", legend=False, ax=axes[1], color="indianred")
axes[1].set_title("PR-AUC by Model")
axes[1].set_ylabel("PR-AUC")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 10. XGBoost Feature Importance

Using the tuned model to see which features drive predictions — including how the frequency-encoded
high-cardinality identifiers (site/app/device) rank against the one-hot low-cardinality features.

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_tuned.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(20)


In [ ]:
top_features = importance_df.head(15)

plt.figure(figsize=(8, 6))
plt.barh(
    top_features["feature"][::-1],
    top_features["importance"][::-1]
)

plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 11. Conclusion

*Fill in with your actual numbers after running the notebook — do not carry over the numbers from the
original version, since the feature set and imbalance handling have both changed.*

Three approaches were evaluated for click-through rate prediction:

| Model | ROC-AUC | PR-AUC | Log Loss |
|---|---:|---:|---:|
| Logistic Regression (balanced) | _fill in_ | _fill in_ | _fill in_ |
| XGBoost (default params, imbalance-aware) | _fill in_ | _fill in_ | _fill in_ |
| XGBoost (tuned) | _fill in_ | _fill in_ | _fill in_ |

Compared to the earlier version of this project, this iteration adds:
- Frequency encoding for high-cardinality identifier features (`site_id`, `app_id`, `device_id`, `device_ip`,
  `device_model`), which the original one-hot-only approach excluded
- Explicit handling of class imbalance via `class_weight` / `scale_pos_weight`, plus PR-AUC as an evaluation
  metric that's more informative than ROC-AUC alone under imbalance
- A small, reproducible randomized hyperparameter search for XGBoost instead of hand-picked parameters

Write 2-3 sentences here on which categorical attributes turned out most predictive, and what that would
imply for ad placement / targeting in practice.
